In [1]:
%cd /home/parthgandhi/Projects/MLBot/mlbot

/home/parthgandhi/Projects/MLBot/mlbot


In [2]:
import polars as pl
import polars.selectors as cs
from src.swing_model.features import gen_features, gen_target
from src.swing_model.loader import gen_scanner_list, gen_stocks_list
from datetime import datetime
import glob

In [3]:
data = pl.scan_parquet("src/test_data.parquet")

In [4]:
START_TIME = datetime(2025, 1, 1)
END_TIME = datetime(2026, 4, 1)

In [5]:
stocks_list = gen_stocks_list(
    files_list=glob.glob("/home/parthgandhi/data/bhavcopy/NSE-Data-bank/data/*.csv"),
    start_date=START_TIME,
    end_date=END_TIME,
)

In [6]:
scanner_stocks = gen_scanner_list(data=data)

In [7]:
scanner_df = (
    scanner_stocks.join(stocks_list, on="timestamp", how="left")
    .filter(pl.col("symbol_count").is_not_null())
    .with_columns(pl.col("symbol").is_in(pl.col("symbol_right")).alias("eq_flag"))
    .filter(pl.col("eq_flag") == True)
    .select("timestamp", "symbol")
    .with_columns(pl.lit(True).alias("scan_flag"))
    .collect()
)

stocks = scanner_df.get_column("symbol").unique().to_list()

In [8]:
len(stocks)

813

In [9]:
ftr = gen_features(data=data).filter(pl.col("symbol").is_in(stocks))
target = gen_target(data=data).filter(pl.col("symbol").is_in(stocks))
res = ftr.join(target, on=["symbol", "timestamp"], how="left")

res = (
    res.with_columns(
        (pl.col("timestamp") - pl.duration(days=1)).alias("timestamp_prev_1")
    )
    .join(
        scanner_df.lazy(),
        left_on=["timestamp_prev_1", "symbol"],
        right_on=["timestamp", "symbol"],
        how="left",
    )
    .filter(pl.col("scan_flag"))
    .select(pl.exclude("scan_flag", "timestamp_prev_1"))
    .drop_nulls()
)

res = res.collect()

In [10]:
print(res.shape)
print(res.get_column("timestamp").min(), res.get_column("timestamp").max())
print(res.get_column("symbol").unique().shape)

(12507, 39)
2025-10-29 2026-04-02
(614,)


In [19]:
res.with_columns(
    pl.col("timestamp").dt.year().alias("year"),
    pl.col("timestamp").dt.month().alias("month"),
).group_by("year", "month", "target").len().sort("year", "month", "target").to_pandas()

,year,month,target,len
0,2025,10,False,590
1,2025,10,True,134
2,2025,11,False,2587
3,2025,11,True,530
4,2025,12,False,1812
5,2025,12,True,489
6,2026,1,False,1729
7,2026,1,True,342
8,2026,2,False,2310
9,2026,2,True,625


In [30]:
res.group_by("target").agg(
    pl.col("std_9_21_dispersion_10").mean().alias("avg_dst_52W")
)

target,avg_dst_52W
bool,f64
false,22.340324
true,22.375927


In [22]:
res.filter(pl.col("target")).sample(10)

symbol,timestamp,is_green_candle,close_position_in_range,roc_3,roc_5,roc_10,log_return_open,log_return_high,log_return_low,log_return_close,dst_from_high_52W,std_9_21,std_9_21_50,body_to_range_pct,upper_wick_to_range_pct,lower_wick_to_range_pct,close_dist_from_close_sma_200,close_sma_50_dist_from_close_sma_200,is_sma_50_gte_sma_200,close_dist_from_close_ema_9,close_dist_from_close_ema_21,close_dist_from_close_sma_50,low_dist_from_close_ema_9,low_dist_from_close_ema_21,low_dist_from_close_sma_50,ma_aligned_bullish,std_9_21_dispersion_5,std_9_21_50_dispersion_5,std_9_21_dispersion_10,std_9_21_50_dispersion_10,std_9_21_dispersion_15,std_9_21_50_dispersion_15,std_9_21_dispersion_20,std_9_21_50_dispersion_20,atr_ratio_14_50,atr_pct_20,adr_pct_20,target
str,date,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
"""APEX""",2025-12-16,true,0.7919,0.0125,0.0038,-0.0741,-0.0369,-0.0163,-0.0053,0.0175,-0.2077,3.464,4.5455,0.526,0.2081,0.2659,0.1152,0.0985,true,-0.0127,-0.0311,0.0153,-0.0384,-0.0563,-0.0112,true,5.1911,6.4278,5.8787,9.1884,6.6327,12.566,8.7559,15.0032,0.9555,0.0519,0.004,true
"""MMFL""",2026-02-18,true,0.8353,0.0144,0.0037,0.0508,0.0227,0.0088,0.0392,0.0286,-0.0078,13.2318,25.1774,0.7017,0.1647,0.1337,0.3338,0.1426,true,0.0333,0.0737,0.1674,-0.0051,0.0339,0.1241,true,12.3757,24.3514,13.4331,24.0927,11.8293,21.86,10.5824,20.4738,1.154,0.0517,0.0022,true
"""CUPID""",2025-12-16,true,0.8226,0.0678,0.076,0.1933,0.0495,0.0119,0.0412,0.0283,0.0,4.9483,9.2815,0.4677,0.1774,0.3548,1.5717,0.8077,true,0.0757,0.168,0.4227,0.0432,0.1327,0.3796,true,4.2645,8.5113,4.2364,8.1472,3.8132,7.4758,3.94,7.2928,1.1153,0.0368,0.0124,true
"""PASHUPATI""",2026-02-17,true,0.8197,0.019,0.0871,0.1303,-0.009,0.0071,0.0012,0.0016,0.0,2.8062,3.3966,0.377,0.1803,0.4426,0.239,0.1156,true,0.045,0.0813,0.1106,-0.0124,0.022,0.0496,true,2.848,3.0596,1.6858,1.8385,1.1876,1.3808,1.0731,1.258,1.2973,0.0471,0.0116,true
"""SAKAR""",2025-11-26,false,0.2967,-0.0153,-0.0777,-0.0837,0.0419,-0.0051,0.0177,-0.0076,-0.0943,5.6107,5.9085,0.6813,0.022,0.2967,0.1513,0.1519,true,-0.0311,-0.0318,-0.0005,-0.0382,-0.0389,-0.0078,true,6.491,8.6588,7.5747,11.8318,8.7069,12.6612,7.2025,10.8633,1.0718,0.0477,0.0029,true
"""RAJRATAN""",2025-12-10,false,0.0941,-0.0291,-0.0085,0.0085,0.0313,-0.006,0.0259,-0.0183,-0.1454,1.8718,14.0685,0.1782,0.7277,0.0941,0.1484,0.0698,true,-0.0099,-0.0078,0.0734,-0.0143,-0.0122,0.0687,true,3.4503,17.1676,2.8201,18.8447,4.5773,21.6162,8.4374,26.703,1.0408,0.0514,0.0024,true
"""KMEW""",2025-12-23,false,0.5547,0.0034,-0.0652,0.0638,0.0919,-0.0988,0.0426,-0.0304,-0.0652,53.914,131.452,0.34,0.1053,0.5547,0.7461,0.4082,true,0.0168,0.0784,0.24,-0.0141,0.0456,0.2022,true,65.9036,138.549,76.3754,137.8761,68.0588,121.5672,59.2173,107.122,1.3642,0.0745,0.0006,true
"""PFOCUS""",2026-02-20,false,0.0789,-0.0518,-0.0099,-0.0131,-0.043,-0.0678,-0.0466,-0.0537,-0.0896,4.4811,10.7479,0.8178,0.1033,0.0789,0.4777,0.3585,true,-0.0317,0.0069,0.0877,-0.0367,0.0017,0.0822,true,8.4321,14.7865,8.8981,15.0489,9.4512,14.702,7.5213,12.5591,1.1575,0.0669,0.0041,true
"""PRECWIRE""",2026-02-13,false,0.1704,0.0096,0.0662,0.1402,0.057,-0.0057,-0.0116,-0.0644,-0.0623,5.2423,8.5849,0.8272,0.0024,0.1704,0.2822,0.1751,true,0.0108,0.0491,0.0911,-0.003,0.0349,0.0763,true,8.3086,10.0071,6.6163,7.378,6.1921,6.8573,6.233,7.3813,1.1442,0.0546,0.004,true
